In [0]:
import sys

silver_path = "/Workspace/Users/himanshu.kumar1@tothenew.com/databricks-medallion-pipeline/src/silver"

if silver_path not in sys.path:
    sys.path.insert(0, silver_path)

import create_silver_tables

print(create_silver_tables.__file__)

/Workspace/Users/himanshu.kumar1@tothenew.com/databricks-medallion-pipeline/src/silver/create_silver_tables.py


In [0]:
result = create_silver_tables.main()

print("Exit code:", result)

if result != 0:
    raise RuntimeError("Silver processing failed")

Starting Silver processing (run_id=a147c198-45cf-456e-9343-8763d7a75945)

--- Silver Processing Summary ---
run_id: a147c198-45cf-456e-9343-8763d7a75945
metrics_rows: 10
  - COMPLETENESS_CUSTOMERS: 99.5% pass (threshold 99.0%, MET)
  - UNIQUENESS_CUSTOMERS: 99.9% pass (threshold 100.0%, NOT MET)
  - TYPE_VALIDATION_CUSTOMERS: 100.0% pass (threshold 99.0%, MET)
  - TYPE_VALIDATION_PRODUCTS: 100.0% pass (threshold 99.0%, MET)
  - BUSINESS_LOGIC_PRODUCTS: 100.0% pass (threshold 99.0%, MET)
  - COMPLETENESS_ORDERS: 99.7% pass (threshold 99.0%, MET)
  - UNIQUENESS_ORDERS: 99.98% pass (threshold 100.0%, NOT MET)
  - TYPE_VALIDATION_ORDERS: 100.0% pass (threshold 99.0%, MET)
  - REFERENTIAL_INTEGRITY_ORDERS: 99.92% pass (threshold 99.9%, MET)
  - BUSINESS_LOGIC_ORDERS: 100.0% pass (threshold 99.0%, MET)

Silver processing completed.
Exit code: 0


In [0]:
%sql
SELECT 'customers' AS entity, COUNT(*) AS row_count
FROM silver.customers

UNION ALL

SELECT 'products', COUNT(*)
FROM silver.products

UNION ALL

SELECT 'orders', COUNT(*)
FROM silver.orders;

entity,row_count
customers,10000
products,500
orders,100000


In [0]:
%sql
SELECT COUNT(*) AS null_email_defects
FROM silver.customers
WHERE email IS NULL
  AND quality_check_result LIKE '%COMPLETENESS%';

null_email_defects
50


In [0]:
%sql
SELECT COUNT(*) AS duplicate_customer_rows
FROM silver.customers
WHERE quality_check_result LIKE '%UNIQUENESS%';

duplicate_customer_rows
10


In [0]:
%sql
SELECT COUNT(*) AS null_customer_id_defects
FROM silver.orders
WHERE customer_id IS NULL
  AND quality_check_result LIKE '%COMPLETENESS%';

null_customer_id_defects
100


In [0]:
%sql
SELECT COUNT(*) AS null_product_id_defects
FROM silver.orders
WHERE product_id IS NULL
  AND quality_check_result LIKE '%COMPLETENESS%';

null_product_id_defects
200


In [0]:
%sql
SELECT COUNT(*) AS orphan_customer_defects
FROM silver.orders
WHERE customer_id IS NOT NULL
  AND customer_id BETWEEN 90001 AND 90050
  AND quality_check_result LIKE '%REFERENTIAL_INTEGRITY%';

orphan_customer_defects
50


In [0]:
%sql
SELECT COUNT(*) AS orphan_product_defects
FROM silver.orders
WHERE product_id IS NOT NULL
  AND product_id BETWEEN 901 AND 930
  AND quality_check_result LIKE '%REFERENTIAL_INTEGRITY%';

orphan_product_defects
30


In [0]:
%sql
SELECT COUNT(*) AS duplicate_order_rows
FROM silver.orders
WHERE quality_check_result LIKE '%UNIQUENESS%';

duplicate_order_rows
20


In [0]:
%sql
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN is_valid = true THEN 1 ELSE 0 END) AS valid_rows,
    SUM(CASE WHEN is_valid = false THEN 1 ELSE 0 END) AS invalid_rows
FROM silver.orders;

total_rows,valid_rows,invalid_rows
100000,99600,400


In [0]:
%sql
SELECT
    run_id,
    entity,
    check_name,
    total_rows,
    passed_rows,
    failed_rows,
    pass_pct,
    threshold_pct,
    threshold_met
FROM silver.dq_metrics
WHERE run_id = 'a147c198-45cf-456e-9343-8763d7a75945'
ORDER BY entity, check_name;

run_id,entity,check_name,total_rows,passed_rows,failed_rows,pass_pct,threshold_pct,threshold_met
a147c198-45cf-456e-9343-8763d7a75945,customers,COMPLETENESS_CUSTOMERS,10000,9950,50,99.50,99.00,true
a147c198-45cf-456e-9343-8763d7a75945,customers,TYPE_VALIDATION_CUSTOMERS,10000,10000,0,100.00,99.00,true
a147c198-45cf-456e-9343-8763d7a75945,customers,UNIQUENESS_CUSTOMERS,10000,9990,10,99.90,100.00,false
a147c198-45cf-456e-9343-8763d7a75945,orders,BUSINESS_LOGIC_ORDERS,100000,100000,0,100.00,99.00,true
a147c198-45cf-456e-9343-8763d7a75945,orders,COMPLETENESS_ORDERS,100000,99700,300,99.70,99.00,true
a147c198-45cf-456e-9343-8763d7a75945,orders,REFERENTIAL_INTEGRITY_ORDERS,100000,99920,80,99.92,99.90,true
a147c198-45cf-456e-9343-8763d7a75945,orders,TYPE_VALIDATION_ORDERS,100000,100000,0,100.00,99.00,true
a147c198-45cf-456e-9343-8763d7a75945,orders,UNIQUENESS_ORDERS,100000,99980,20,99.98,100.00,false
a147c198-45cf-456e-9343-8763d7a75945,products,BUSINESS_LOGIC_PRODUCTS,500,500,0,100.00,99.00,true
a147c198-45cf-456e-9343-8763d7a75945,products,TYPE_VALIDATION_PRODUCTS,500,500,0,100.00,99.00,true
